# Binary Classification

In [ ]:
import os
import logging
import warnings
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib as mpl
import matplotlib.pyplot as plt
import shap

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, RepeatedStratifiedKFold, GridSearchCV
from statsmodels.stats.outliers_influence import variance_inflation_factor
from xgboost import XGBClassifier

In [ ]:
from utils.constants import RANDOM_SEED
from utils.common import (
    get_data_folder_path,
    set_plot_font_sizes,
    plot_boxplot_by_class,
    plot_correlation_matrix,
)
from utils.classification import (
    describe_input_features,
    plot_confusion_matrix,
    plot_target_rate,
    compute_classification_metrics,
    build_coefficients_table,
    plot_coefficients_values,
    plot_coefficients_significance,
    plot_eval_metrics_xgb,
    plot_gain_metric_xgb,
    plot_shap_importance,
    plot_shap_beeswarm,
    build_ks_table,
    beautify_ks_table,
    plot_ks_table,
    plot_roc_curve,
)
from utils.feature_selection import run_feature_selection_steps

In [ ]:
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%H:%M:%S",
)
logger = logging.getLogger(__name__)

pd.set_option("display.max_columns", None)
pd.options.display.float_format = "{:.2f}".format

mpl.rcParams["font.sans-serif"] = "Arial"
plt.set_loglevel("WARNING")

# plots configuration
sns.set_style("darkgrid")
sns.set_palette("colorblind")
set_plot_font_sizes()
%matplotlib inline

## 1. Define Parameters

In [ ]:
target_col = "is_normal"
target_classes_dict = {
    0: "Not Normal",
    1: "Normal"
}
test_size = 0.20

## 2. Load Data

In this notebook we will use the Fetal Health Dataset.

Sources:
1. Fetal Health Classification Dataset: https://www.kaggle.com/datasets/andrewmvd/fetal-health-classification
2. Original article: https://www.ncbi.nlm.nih.gov/pmc/articles/PMC68223152

In [ ]:
data_path = get_data_folder_path()

df_input = pd.read_csv(os.path.join(data_path, "fetal_health.csv"))
df_input.columns = [col.replace(" ", "_") for col in df_input.columns]

### Target column

Fetal health (target column) can have the following values:
- 1 - Normal
- 2 - Suspect
- 3 - Pathological

For this notebook, we will consider the Normal/not Normal distinction for binary classification

In [ ]:
df_input[target_col] = (df_input["fetal_health"] == 1).astype(np.int8)
df_input.drop(columns=["fetal_health"], inplace=True)

### Train test split

In [ ]:
df_input_train, df_input_test = train_test_split(
    df_input,
    test_size=test_size,
    stratify=df_input[target_col],
    random_state=RANDOM_SEED,
)

In [ ]:
pd.concat([
    df_input_train[target_col].value_counts(dropna=False, normalize=False).rename("train_target_count"),
    df_input_train[target_col].value_counts(dropna=False, normalize=True).rename("train_target_pct"),
    df_input_test[target_col].value_counts(dropna=False, normalize=False).rename("test_target_count"),
    df_input_test[target_col].value_counts(dropna=False, normalize=True).rename("test_target_pct"),
], axis=1)

In [ ]:
describe_input_features(df_input, df_input_train, df_input_test)

## 3. Exploratory Data Analysis (EDA)

### Boxplots by Target Class

In [ ]:
fig = plot_boxplot_by_class(
    df=df_input_train,  # use only training data to avoid bias in test results
    class_col=target_col,
    plots_per_line=6,
    title="Features in input dataset",
)

### Pearson's Correlation

In [ ]:
fig = plot_correlation_matrix(
    df=df_input_train,  # use only training data to avoid bias in test results
    method="pearson",
    fig_height=10
)

## 4. Feature Selection

In [ ]:
fs_steps = {
    "manual": dict(
        cols_to_exclude=[
            "histogram_variance",
            "severe_decelerations",
        ]
    ),
    "variance": dict(threshold=0),
    "correlation": dict(threshold=0.9),
    "l1_regularization": dict(
        problem="classification",
        train_test_split_params=dict(test_size=test_size),
        logspace_search=dict(start=-5, stop=1, num=20, base=10),
        # tolerance over minimum error with which to search for the best model
        error_tolerance_pct=0.02,
        # minimum features to keep in final selection
        min_feats_to_keep=4,
        random_seed=RANDOM_SEED,
    ),
    "vif": dict(threshold=5),
}

In [ ]:
selected_feats, df_fs = run_feature_selection_steps(
    df_input=df_input_train,  # use only training data to avoid bias in test results
    target_col=target_col,
    fs_steps=fs_steps
)

### Correlation check


In [ ]:
fig = plot_correlation_matrix(
    df=df_input_train[selected_feats + [target_col]],  # use only training data to avoid bias in test results
    method="pearson",
    fig_height=5
)

### Multicollinearity check

Multicollinearity is a problem because it undermines the statistical significance of an independent variable. [\[source\]](https://link.springer.com/chapter/10.1007/978-0-585-25657-3_37)

Multicollinearity does not affect the accuracy of predictive models, including regression models. \[...\] Now, where multicollinearity becomes 'an issue' is when you want to 'interpret' the parameters learned by your model. In other words, you cannot say that the feature with the 'biggest weight' is 'the most important' when the features are correlated. Note that this is independent on the accuracy of the model, this is only the interpretation part [\[source\]](https://www.researchgate.net/post/Are-Random-Forests-affected-by-multi-collinearity-between-features)



**Variance Inflation Factor (VIF)**

The variance inflation factor (VIF) is a statistical tool that measures the amount of multicollinearity in a regression model. As a general rule of thumb, "VIF > 5 is cause for concern and VIF > 10 indicates a serious collinearity problem."

The higher the VIF:
- The more correlated a predictor is with the other predictors
- The more the standard error is inflated
- The larger the confidence interval
- The less likely it is that a coefficient will be evaluated as statistically significant
[\[source\]](https://towardsdatascience.com/everything-you-need-to-know-about-multicollinearity-2f21f082d6dc)

In [ ]:
df_vif = pd.DataFrame(
    data=[variance_inflation_factor(df_input_train[selected_feats].values, i) for i in range(len(selected_feats))],
    index=selected_feats,
    columns=["VIF"]
).sort_values("VIF", ascending=False)

df_vif

### Model input datasets

In [ ]:
# train datasets
X_train = df_input_train[selected_feats]
y_train = df_input_train[target_col]
# test datatsets
X_test = df_input_test[selected_feats]
y_test = df_input_test[target_col]

### Scaling

In [ ]:
# Standardize X_train and X_test
stdscaler = StandardScaler()
X_train_std = pd.DataFrame(stdscaler.fit_transform(X_train), columns=X_train.columns, index=X_train.index)
X_test_std = pd.DataFrame(stdscaler.transform(X_test), columns=X_test.columns, index=X_test.index)

## 5. Classifier Model

### Select classifier: Logistic Regression or XGBoost

In [ ]:
MODEL_SELECTION = "logistic_regression"
# MODEL_SELECTION = "xgboost"

### Hyperparameter tuning with K-Fold Cross Validation

For detailed explanation on XGBoost's parameters: https://www.kaggle.com/code/prashant111/a-guide-on-xgboost-hyperparameters-tuning/notebook

In [ ]:
if MODEL_SELECTION == "logistic_regression":
    Estimator = LogisticRegression
    cv_search_space = {
        "penalty": ["l1", "l2", "elasticnet"],
        "C": np.logspace(-3, 1, num=9, base=10.0),
        "class_weight": [None],
    }
elif MODEL_SELECTION == "xgboost":
    Estimator = XGBClassifier
    cv_search_space = {
        "objective": ["binary:logistic"],
        "n_estimators": [30, 40, 50],
        "learning_rate": [0.1],
        "max_depth": [3, 4, 6],
        "min_child_weight": [2, 4],
        "gamma": [0, 0.5],
        "alpha":[0, 0.3],
        "scale_pos_weight": [1],
        "lambda":[1],
        ## "subsample": [0.8, 1.0],
        ## "colsample_bytree": [0.8, 1.0],
        "verbosity": [0],
    }
else:
    raise ValueError(
        "'MODEL_SELECTION' must be either 'logistic_regression' or 'xgboost'. "
        f"Got {MODEL_SELECTION} instead."
    )

In [ ]:
cv_scoring_metrics = {
    "roc_auc": "ROC AUC",
    "accuracy": "Accuracy",
    "precision": "Precision",
    "recall": "Recall",
    "f1": "F1 Score",
}

In [ ]:
%%time
# define evaluation
kfold_cv = RepeatedStratifiedKFold(n_splits=3, n_repeats=1, random_state=RANDOM_SEED)
# define search
grid_search = GridSearchCV(
    estimator=Estimator(),
    param_grid=cv_search_space,
    scoring=list(cv_scoring_metrics.keys()),
    cv=kfold_cv,
    refit="f1",
    verbose=1,
)
# execute search
with warnings.catch_warnings(action="ignore"):
    result_cv = grid_search.fit(X_train, y_train)

In [ ]:
print("Grid Search CV Best Model - Scoring Metrics:")
for i, (metric_key, metric_name) in enumerate(cv_scoring_metrics.items(), start=1):
    print(
        f"  {i}. {(metric_name + ":").ljust(10)} "
        f"{result_cv.cv_results_[f"mean_test_{metric_key}"][result_cv.best_index_]:.3f}"
    )
print(f"\nBest Hyperparameters: {result_cv.best_params_}")

### Final Model

In [ ]:
# instantiate model with best hyperparameters and additional kwargs
if MODEL_SELECTION == "logistic_regression":
    model_kwargs = dict()
    model_fit_kwargs = dict()
elif MODEL_SELECTION == "xgboost":
    eval_metrics = dict(
        logloss="Binary Cross-entropy Loss (Log-loss)",
        error="Binary Classification Error Rate",
        auc="ROC AUC",
    )
    model_kwargs = dict(eval_metric=list(eval_metrics.keys()))
    model_fit_kwargs = dict(
        eval_set=[(X_train_std, y_train), (X_test_std, y_test)],
        verbose=False
    )
    
model = Estimator(**result_cv.best_params_, **model_kwargs, random_state=RANDOM_SEED)

In [ ]:
# Fit model and make predictions
model.fit(X_train_std, y_train, **model_fit_kwargs)
# Make predictions
y_pred_proba_train = pd.Series(data=model.predict_proba(X_train_std)[:, 1], index=X_train_std.index)
y_pred_proba = pd.Series(data=model.predict_proba(X_test_std)[:, 1], index=X_test_std.index)

In [ ]:
if MODEL_SELECTION == "xgboost":
    fig = plot_eval_metrics_xgb(model.evals_result(), eval_metrics)

**Plot target rate per group of predicted probability**

A good model should have increasing target rate for each group of predicted probability (e.g. quartiles, deciles)

In [ ]:
fig = plot_target_rate(y_test, y_pred_proba)

**Define optimal threshold for separating classes using the ROC Curve**

The optimal threshold is the one that maximizes the difference between the True Positive Rate (TPR) and False Positive Rate (FPR).

In [ ]:
fig, optimal_thresh = plot_roc_curve(
    y_true=y_train,  # use only training data to avoid bias in test results
    y_pred_proba=y_pred_proba_train,
    title="ROC Curve on Training Data",
    ret_optimal_thresh=True
)

In [ ]:
# compute binary predictions
print(f"Optimal Threshold for Classification: {100*optimal_thresh:.2f}%")
y_pred_train = (y_pred_proba_train > optimal_thresh).astype(int)
y_pred = (y_pred_proba > optimal_thresh).astype(int)

### Feature Importance

- For Logistic Regression: coefficients values and statistical significance
- For XGBoost: SHAP analysis and Gain Metric

In [ ]:
if MODEL_SELECTION == "logistic_regression":
    df_coefficients = build_coefficients_table(model, X_train_std)
    fig = plot_coefficients_values(df_coefficients)
    fig = plot_coefficients_significance(df_coefficients, log_scale=False)
    
elif MODEL_SELECTION == "xgboost":
    # compute SHAP values
    explainer = shap.Explainer(model)
    shap_values = explainer(X_test_std)
    # shap plots
    fig = plot_shap_importance(shap_values)
    fig = plot_shap_beeswarm(shap_values)
    fig = plot_gain_metric_xgb(model, X_test_std)

### Performance Metrics

In [ ]:
df_train_metrics = pd.Series(
    compute_classification_metrics(y_train, y_pred_train, y_pred_proba_train)
).to_frame(name="Train Metrics")
df_test_metrics = pd.Series(
    compute_classification_metrics(y_test, y_pred, y_pred_proba)
).to_frame(name="Test Metrics")

print("Final Model - Scoring Metrics on Train & Test Datasets:")
df_metrics = df_train_metrics.join(df_test_metrics)
display(df_metrics)

#### Confusion Matrix

In [ ]:
# Confusion Matrix
fig = plot_confusion_matrix(
    y_test,
    y_pred,
    estimator=model,
    target_classes_dict=target_classes_dict,
    normalize="true",
)

#### ROC AUC

In [ ]:
fig = plot_roc_curve(y_test, y_pred_proba)

#### KS Gain

In [ ]:
df_ks, ks_score = build_ks_table(y_test, y_pred_proba, ret_ks=True)
print(f"KS score: {ks_score * 100:.2f} p.p.")
beautify_ks_table(df_ks)

In [ ]:
fig = plot_ks_table(df_ks)